# Park其餘26段｜九句全接受後才能生成
同一私人park-voice-inputs.zip。先核對九句正式結果ZIP的accepted驗收。生成26，原九句與Library共用音鎖定；最終36段。未接受九句不會載入模型。

## 1｜安裝（免費T4）

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2｜上傳相同Park私人輸入包

In [ ]:
from google.colab import files, output as colab_output
from pathlib import Path
import uuid, zipfile, io, hashlib, importlib.util, sys
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('請只上傳 park-voice-inputs.zip')
raw=next(iter(uploaded.values()))
# Read only the known tool entry before importing code from the verified input package.
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if sum(i.file_size for i in z.infolist())>100000000 or len(z.infolist())>300:raise ValueError('輸入ZIP過大')
    if z.namelist().count('park_tools.py')!=1:raise ValueError('輸入工具檔不符')
    tool_bytes=z.read('park_tools.py')
if hashlib.sha256(tool_bytes).hexdigest()!='30e82f59aaec5edccb78b055d42bcf65e6d36df26702477e99226d503388cb43':raise ValueError('工具程式hash不符，請使用這次提供的輸入ZIP')
spec=importlib.util.spec_from_loader('park_voice_current',loader=None)
park=importlib.util.module_from_spec(spec)
exec(compile(tool_bytes,'park_tools.py','exec'),park.__dict__)
WORK=park.unpack(raw,Path('/content')/('park_inputs_'+uuid.uuid4().hex[:8]))
park.validate_work(WORK)
colab_output.enable_custom_widget_manager()
from IPython.display import display,Markdown,Audio
OUTPUT=Path('/content')/('park_trial9_'+uuid.uuid4().hex[:8])
print('核准Rosie/Pirate主參考、逐字稿及Library共用音SHA256已核對；未生成。')

## 3｜上傳九句已接受的結果ZIP

In [ ]:
nine_upload=files.upload()
if len(nine_upload)!=1:raise ValueError('只選一個九句結果ZIP')
ACCEPTED_TRIAL=park.restore_results(next(iter(nine_upload.values())),Path('/content'))
park.approved_trial(ACCEPTED_TRIAL)
OUTPUT=Path('/content')/('park_complete36_'+uuid.uuid4().hex[:8])
print('九句WAV及accepted紀錄已核對，全部將原樣保留。')

## 4｜其餘26句dry-run（Rosie10、Pirate16）

In [ ]:
dry=park.generate(WORK,OUTPUT,'remaining26',run=False,accepted_trial=ACCEPTED_TRIAL)
for a in dry['jobs']:print(a['lineId'],a['role'],a['voiceId'],a['text'])
print('26待生成，generated=0；九句+共用1將保留原WAV。')

## 5｜26段生成（待家長確認九句品質後手動開啟）

In [ ]:
RUN_GENERATION=False
CONFIRMATION='' # CONFIRM_PARK_REMAINING_26
if RUN_GENERATION:
    print(park.generate(WORK,OUTPUT,'remaining26',run=True,confirmation=CONFIRMATION,accepted_trial=ACCEPTED_TRIAL))
else:print('尚未生成26句。')

## 6｜最終36段驗收，九句與共用1鎖定

In [ ]:
if (OUTPUT/'generated-manifest.json').exists():
    PARK_REVIEW_UI=park.show_review(OUTPUT)
else:print('沒有輸出；此格不生成。')

## 7｜可選：上傳26段批次的部分或完整結果ZIP續跑／重開驗收

In [ ]:
resume_upload=files.upload()
if len(resume_upload)!=1:raise ValueError('只上傳一個結果ZIP')
OUTPUT=park.restore_results(next(iter(resume_upload.values())),Path('/content'))
if park.read(OUTPUT/'generated-manifest.json')['stage']!='remaining26':raise ValueError('需26段批次的結果ZIP')
PARK_REVIEW_UI=park.show_review(OUTPUT)
print('續跑還需本執行階段WORK與ACCEPTED_TRIAL；重新執行第5格只生成尚缺句。')